Notebook ini merapikan data film untuk proyek Sains Data MovieMatch. Data ini dipakai pada sistem rekomendasi berbasis konten (TF-IDF). Hasilnya adalah file `movies_clean_final.csv`, yang dibagikan lewat Google Drive karena ukurannya terlalu besar untuk GitHub. Data diambil dari [The Movies Dataset](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset) oleh Rounak Banik di Kaggle. Isinya data sekitar 45.000 film dari TMDB, umumnya sampai tahun 2017.

Dari file ZIP, hanya tiga file yang dipakai:

- `movies_metadata.csv`: data utama film
- `keywords.csv`: kata kunci cerita
- `credits.csv`: pemain dan kru, termasuk sutradara

File `links.csv` dan `ratings.csv` tidak dipakai karena model ini menilai kemiripan dari isi film, bukan dari rating pengguna.

**Cara menjalankan di Google Colab:**

1. Unduh ZIP dari Kaggle (sekitar 228 MB). Nama filenya boleh apa saja.
2. Unggah ZIP ke Google Drive, karena unggahan langsung ke Colab sering terputus.
3. Salin ZIP ke folder `data/`.
4. Jalankan semua sel lewat **Runtime → Run all**. File hasilnya bisa diunduh di sel terakhir.

**Di komputer sendiri:** letakkan ZIP di folder `data/`, lalu jalankan semua sel. Pastikan `pandas`, `matplotlib`, dan `numpy` sudah terpasang.

**Susunan foldernya:**

```text
proyek/
├── data/                          ← taruh file ZIP dataset di sini
│   └── the-movies-dataset.zip     ← nama file bebas
├── sample_data
└── movies_clean_final.csv         ← hasil akhir (dibagikan lewat Google Drive)
```

Arti tiap kolom:

- `id`: nomor film di TMDB, dipakai untuk menggabungkan tabel.
- `title`: judul film.
- `overview`: sinopsis film. Jika tidak ada, disimpan kosong dan filmnya tetap disimpan.
- `genres_list`: daftar genre, dipisah tanda `|` (contoh: `Drama|Crime`).
- `keywords_list`: daftar kata kunci, dipisah tanda `|`.
- `director`: nama sutradara, dipisah tanda `|` jika lebih dari satu.
- `release_year`: tahun rilis. Boleh kosong jika tanggal aslinya tidak valid.
- `vote_average`: rating rata-rata TMDB (skala 0–10).
- `vote_count`: jumlah orang yang memberi rating.

Urutan kolom di file hasil mengikuti daftar di atas.

## 1. Pengaturan pustaka dan path data

Tahap ini menyiapkan pustaka dan lokasi file data. Data disimpan di folder `data/`, dan nama file ZIP boleh apa saja.

Notebook mencari file ZIP dengan urutan berikut:

- Jika `ZIP_PATH` diisi, file itu yang dipakai.
- Jika tidak, notebook mencari file `.zip` di folder `data/`, lalu di folder kerja.
- Jika file tetap tidak ditemukan dan notebook berjalan di Colab, akan muncul tombol unggah.

Di tahap ini juga disiapkan fungsi `jumlah_teks_kosong`. Fungsi ini dipakai untuk menghitung sinopsis yang kosong, baik berupa NaN, teks kosong, maupun yang hanya berisi spasi.


In [ ]:
from pathlib import Path
import ast
import os
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Menentukan folder utama repo agar path tetap benar saat notebook dijalankan dari folder notebooks/
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = ROOT_DIR / "data" / "raw"
EXTRACT_DIR = DATA_DIR / "extracted"
CSV_OUTPUT = ROOT_DIR / "data" / "processed" / "movies_clean_final.csv"
CSV_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
ZIP_PATH = None  # contoh: DATA_DIR / "the-movies-dataset.zip"


def jumlah_teks_kosong(series):
    """Menghitung nilai yang NaN, string kosong, atau hanya spasi."""
    return int(series.fillna("").astype(str).str.strip().eq("").sum())


def temukan_zip(zip_path=ZIP_PATH):
    """Mencari file ZIP dataset di ZIP_PATH, folder data/, atau folder kerja."""
    if zip_path is not None:
        p = Path(zip_path)
        if not p.exists():
            raise FileNotFoundError(f"File ZIP tidak ditemukan: {p}")
        return p

    kandidat = []
    if DATA_DIR.is_dir():
        kandidat.extend(sorted(DATA_DIR.glob("*.zip")))
    kandidat.extend(sorted(Path(".").glob("*.zip")))

    unik = []
    sudah = set()
    for file_zip in kandidat:
        kunci = file_zip.resolve()
        if kunci not in sudah:
            sudah.add(kunci)
            unik.append(file_zip)

    if len(unik) == 1:
        return unik[0]
    if len(unik) > 1:
        print("Ditemukan lebih dari satu file ZIP:")
        for i, file_zip in enumerate(unik, 1):
            print(f"  {i}. {file_zip}")
        raise SystemExit(
            "Isi ZIP_PATH di sel pengaturan path dengan salah satu path di atas, "
            "lalu jalankan ulang sel."
        )
    return None


def path_csv(nama_file):
    """Menemukan file CSV hasil ekstraksi, termasuk jika ada folder di dalam ZIP."""
    langsung = EXTRACT_DIR / nama_file
    if langsung.exists():
        return langsung
    hits = list(EXTRACT_DIR.rglob(nama_file))
    if not hits:
        raise FileNotFoundError(
            f"{nama_file} tidak ditemukan di {EXTRACT_DIR}. "
            "Periksa apakah file ZIP-nya benar."
        )
    return hits[0]


DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_file = temukan_zip()

if zip_file is None:
    try:
        from google.colab import files

        print("File ZIP belum ditemukan. Silakan unggah dataset (file .zip).")
        uploaded = files.upload()
        nama_zip = [nama for nama in uploaded if nama.lower().endswith(".zip")]
        if not nama_zip:
            raise FileNotFoundError("Tidak ada file .zip yang diunggah.")
        if len(nama_zip) > 1:
            print("Beberapa file ZIP diunggah; yang dipakai:", nama_zip[0])
        # Pindahkan ke data/ agar path-nya rapi dan bisa dipakai ulang.
        tujuan = DATA_DIR / Path(nama_zip[0]).name
        Path(nama_zip[0]).replace(tujuan)
        zip_file = tujuan
    except ImportError as exc:
        raise FileNotFoundError(
            "File ZIP tidak ditemukan. Letakkan file .zip di folder data/ "
            "atau di folder kerja, atau isi ZIP_PATH."
        ) from exc

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print(f"Dataset diekstrak dari : {zip_file}")
print(f"Folder tujuan          : {EXTRACT_DIR.resolve()}")


File ZIP belum ditemukan. Silakan unggah dataset (file .zip).


## 2. Mengecek isi ZIP

Setelah ZIP diekstrak, tahap ini memastikan tiga file utama tersedia, yaitu `movies_metadata.csv`, `keywords.csv`, dan `credits.csv`. Ketiga file ini dipakai untuk menyusun CSV bersih.


In [ ]:
file_csv = sorted(EXTRACT_DIR.rglob("*.csv"))
print(f"Jumlah file CSV: {len(file_csv)}")
for file in file_csv:
    print(file.relative_to(EXTRACT_DIR))


## 3. Membaca dataset

Tahap ini membaca tiga file utama. File `ratings.csv` tidak dibaca karena ukurannya sangat besar dan tidak dibutuhkan oleh model content-based yang dikerjakan.


In [ ]:
df_movies = pd.read_csv(path_csv("movies_metadata.csv"), low_memory=False)
df_keywords = pd.read_csv(path_csv("keywords.csv"))
df_credits = pd.read_csv(path_csv("credits.csv"))

print("Semua dataset utama berhasil dibaca.")
print("Movies   :", df_movies.shape)
print("Keywords :", df_keywords.shape)
print("Credits  :", df_credits.shape)


## 4. Inspeksi awal

Sebelum data digabungkan, tahap ini mengecek kondisi tiap tabel. Yang dicek adalah ukuran data, nama kolom, tipe data, jumlah nilai kosong, dan baris yang sama persis.

Untuk kolom `overview`, `isnull()` hanya menghitung nilai NaN. Sinopsis yang hanya berisi spasi belum ikut terhitung, jadi pengecekan lengkapnya dilakukan di tahap berikutnya.


In [ ]:
datasets = {
    "Movies Metadata": df_movies,
    "Keywords": df_keywords,
    "Credits": df_credits,
}

for nama, df in datasets.items():
    print(f"\n===== {nama} =====")
    print("Jumlah baris :", df.shape[0])
    print("Jumlah kolom :", df.shape[1])
    print("Nama kolom   :", list(df.columns))

print("\n===== INFORMASI DATASET FILM =====")
df_movies.info()

print("\n===== JUMLAH NILAI KOSONG (Movies) =====")
print(df_movies.isnull().sum())

print("\n===== BARIS IDENTIK (duplikat seluruh kolom) =====")
for nama, df in datasets.items():
    print(f"{nama}: {df.duplicated().sum()}")


## 5. Menghitung jumlah overview kosong

Tahap ini menghitung film yang tidak punya sinopsis. Jika hanya menghitung NaN, jumlahnya 954. Setelah NaN diganti teks kosong, `.isna()` menunjukkan 0, padahal sinopsisnya tetap tidak ada.

Ada juga 5 film yang sinopsisnya hanya berisi spasi. Karena itu, yang dihitung adalah NaN, teks kosong, dan teks berisi spasi, sehingga totalnya 954 + 5 = 959 film.

Film-film ini tidak dihapus, karena judul, genre, dan kata kuncinya masih berguna untuk model. NaN cukup diganti teks kosong agar proses TF-IDF tidak error.


In [ ]:
nan_saja = int(df_movies["overview"].isna().sum())
kosong_semua = jumlah_teks_kosong(df_movies["overview"])
hanya_spasi = (
    (~df_movies["overview"].isna())
    & (df_movies["overview"].astype(str).str.strip() == "")
)

print("Overview NaN saja                    :", nan_saja)
print("Overview kosong/hanya spasi          :", kosong_semua)
print("Selisih (teks hanya spasi, bukan NaN):", kosong_semua - nan_saja)

print("\nFilm yang overview-nya hanya spasi:")
display(
    df_movies.loc[
        hanya_spasi,
        ["id", "title", "overview", "release_date"],
    ]
)


## 6. Pemeriksaan ID

ID film dipakai untuk menggabungkan metadata, kata kunci, dan credits. Karena itu, ID harus berupa angka, tidak boleh dobel, dan harus cocok di ketiga tabel.

Dari pemeriksaan ini, ditemukan dua masalah:

- Beberapa ID di `movies_metadata` berisi tanggal, bukan angka. Baris ini kolomnya bergeser, jadi bukan data film yang bisa dipakai.
- Beberapa ID muncul lebih dari sekali. Ada yang isinya sama persis, ada juga yang hanya berbeda di kolom seperti `popularity` atau `vote_count`.

In [ ]:
print("===== PEMERIKSAAN ID =====")
for nama, df in datasets.items():
    id_numerik = pd.to_numeric(df["id"], errors="coerce")
    print(f"\n{nama}")
    print("  Jumlah baris        :", len(df))
    print("  ID unik             :", df["id"].nunique())
    print("  ID duplikat         :", int(df["id"].duplicated().sum()))
    print("  ID tidak valid      :", int(id_numerik.isna().sum()))

id_numerik_movies = pd.to_numeric(df_movies["id"], errors="coerce")
data_id_invalid = df_movies[id_numerik_movies.isna()]

print("\n===== BARIS DENGAN ID TIDAK VALID =====")
display(
    data_id_invalid[
        ["id", "title", "genres", "overview", "release_date", "runtime"]
    ]
)


## 7. Alasan penanganan data kosong dan duplikat

Cara membersihkan data di tahap ini mengikuti hasil pemeriksaan sebelumnya.

**Data yang dihapus:**

- **ID tidak valid.** Contohnya ID berisi tanggal seperti `1997-08-20`. Baris ini tidak punya judul dan kolomnya bergeser, jadi bukan data film.
- **Baris yang sama persis.** Baris seperti ini hanya mengulang film yang sama.
- **ID dobel yang isinya sedikit berbeda,** biasanya di `popularity` atau `vote_count`. Hanya satu baris yang disimpan. Data diurutkan berdasarkan `vote_count` terbesar dengan pengurutan yang stabil, lalu baris pertama yang dipakai, supaya yang disimpan adalah baris dengan jumlah vote tertinggi dan hasilnya selalu sama setiap kali notebook dijalankan. Setelah baris terpilih, data diurutkan kembali berdasarkan `id`. Judul dan sinopsisnya sama, jadi tidak ada teks penting yang hilang.
- **Film tanpa judul** (3 film) dihapus di akhir, karena judul dibutuhkan untuk menampilkan rekomendasi. Ketiga film ini punya sinopsis, jadi jumlah overview kosong tetap 959.

**Data yang tetap disimpan:**

- **Film tanpa sinopsis.** NaN diganti teks kosong pada tahap pembersihan. Sebelum data disimpan, spasi di awal dan akhir sinopsis dibuang, lalu sinopsis yang menjadi kosong diubah menjadi nilai kosong (NaN). Dengan cara ini, hitungan overview kosong tetap 959, baik di notebook maupun saat file CSV dibaca ulang.
- **Film tanpa tahun rilis, genre, kata kunci, atau sutradara.** Nilainya dikosongkan, karena data lainnya masih bisa dipakai. Tahun rilis yang tidak valid juga menjadi kosong.

**Untuk keywords dan credits:** tipe ID disamakan, baris yang sama persis dihapus, dan setiap ID hanya punya satu baris. Jika ada data credits ganda untuk ID yang sama, data krunya digabung supaya nama sutradara tidak hilang.


In [ ]:
# Cuplikan bukti: ID movies yang masih berulang setelah baris identik dibuang
movies_temp = df_movies.copy()
movies_temp["id"] = pd.to_numeric(movies_temp["id"], errors="coerce")
movies_temp = movies_temp.dropna(subset=["id"]).drop_duplicates()

dup_movies = movies_temp[movies_temp["id"].duplicated(keep=False)].sort_values("id")
print("Jumlah baris dengan ID berulang (setelah buang baris identik):", len(dup_movies))
print("Jumlah ID yang berulang :", dup_movies["id"].nunique())

variasi = movies_temp.groupby("id").nunique(dropna=False)
id_berbeda = variasi[(variasi > 1).any(axis=1)]
print("Jumlah ID yang atributnya tidak seluruhnya sama:", len(id_berbeda))
display(id_berbeda.head(15))


## 8. Membersihkan movies, keywords, dan credits

Tahap ini membersihkan ketiga tabel. Pembersihan dilakukan pada salinan data, jadi tabel aslinya tetap ada untuk dibandingkan. Setelah tahap ini, setiap ID yang valid hanya punya satu baris di tiap tabel.


In [ ]:
# ----- Movies -----
movies_clean = df_movies.copy()
movies_clean["id"] = pd.to_numeric(movies_clean["id"], errors="coerce")
movies_clean = movies_clean.dropna(subset=["id"])
movies_clean["id"] = movies_clean["id"].astype("int64")
movies_clean = movies_clean.drop_duplicates()
# Mengurutkan berdasarkan vote_count terbesar (pengurutan stabil)
# agar baris dengan jumlah vote tertinggi yang disimpan
movies_clean = movies_clean.sort_values(
    by="vote_count", ascending=False, kind="mergesort"
)
movies_clean = movies_clean.drop_duplicates(subset="id", keep="first")
# Mengembalikan urutan berdasarkan id setelah baris terpilih
movies_clean = movies_clean.sort_values(by="id").reset_index(drop=True)
# Isi NaN menjadi teks kosong; filmnya tetap disimpan
movies_clean["overview"] = movies_clean["overview"].fillna("")

print("===== MOVIES SETELAH CLEANING =====")
print("Jumlah baris   :", len(movies_clean))
print("ID unik        :", movies_clean["id"].nunique())
print("ID duplikat    :", int(movies_clean["id"].duplicated().sum()))
print(
    "Overview kosong (NaN/teks kosong/hanya spasi):",
    jumlah_teks_kosong(movies_clean["overview"]),
)

# ----- Keywords -----
keywords_clean = df_keywords.copy()
keywords_clean["id"] = pd.to_numeric(keywords_clean["id"], errors="coerce")
keywords_clean = keywords_clean.dropna(subset=["id"])
keywords_clean["id"] = keywords_clean["id"].astype("int64")
keywords_clean = keywords_clean.drop_duplicates()
keywords_clean = keywords_clean.drop_duplicates(subset=["id"], keep="first")

print("\n===== KEYWORDS SETELAH CLEANING =====")
print("Jumlah baris :", len(keywords_clean))
print("ID unik      :", keywords_clean["id"].nunique())
print("ID duplikat  :", int(keywords_clean["id"].duplicated().sum()))


# ----- Credits -----
def parse_list(data):
    """Mengubah teks yang menyerupai list Python menjadi list sungguhan."""
    try:
        hasil = ast.literal_eval(data)
        return hasil if isinstance(hasil, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []


def gabung_crew(kumpulan):
    """Menggabungkan data kru dari beberapa baris dengan ID yang sama."""
    hasil = {}
    for crew in kumpulan:
        for orang in parse_list(crew):
            if isinstance(orang, dict):
                kunci = orang.get("credit_id") or str(orang)
                hasil[kunci] = orang
    return str(list(hasil.values()))


credits_temp = df_credits.copy()
credits_temp["id"] = pd.to_numeric(credits_temp["id"], errors="coerce")
credits_temp = credits_temp.dropna(subset=["id"])
credits_temp["id"] = credits_temp["id"].astype("int64")

print("\nMenggabungkan credits per ID (perlu waktu sejenak)...")
credits_clean = credits_temp.groupby("id", as_index=False).agg(
    {"cast": "first", "crew": gabung_crew}
)

print("===== CREDITS SETELAH CLEANING =====")
print("Jumlah baris :", len(credits_clean))
print("ID unik      :", credits_clean["id"].nunique())
print("ID duplikat  :", int(credits_clean["id"].duplicated().sum()))


## 9. Ringkasan hasil cleaning dan kesesuaian ID

Tahap ini menunjukkan berapa baris yang berkurang di tiap tabel setelah dibersihkan. Selain itu, dihitung juga berapa film di `movies` yang tidak punya pasangan di `keywords` atau `credits`. Saat tabel digabungkan nanti, dipakai `how='left'` supaya film tanpa pasangan tetap tersimpan.


In [ ]:
hasil_cleaning = pd.DataFrame(
    {
        "Dataset": ["Movies Metadata", "Keywords", "Credits"],
        "Sebelum Cleaning": [len(df_movies), len(df_keywords), len(df_credits)],
        "Sesudah Cleaning": [len(movies_clean), len(keywords_clean), len(credits_clean)],
    }
)
hasil_cleaning["Data Dihapus"] = (
    hasil_cleaning["Sebelum Cleaning"] - hasil_cleaning["Sesudah Cleaning"]
)
display(hasil_cleaning)

id_movies = set(movies_clean["id"])
id_keywords = set(keywords_clean["id"])
id_credits = set(credits_clean["id"])

print("ID Movies   :", len(id_movies))
print("ID Keywords :", len(id_keywords))
print("ID Credits  :", len(id_credits))
print("ID Movies tanpa Keywords :", len(id_movies - id_keywords))
print("ID Movies tanpa Credits  :", len(id_movies - id_credits))
print(
    "ID yang ada di ketiga dataset:",
    len(id_movies & id_keywords & id_credits),
)


## 10. Menggabungkan metadata, kata kunci, dan credits

Tahap ini menggabungkan ketiga tabel berdasarkan `id`. Dipakai `how='left'` supaya semua film di `movies_clean` tetap ada, walaupun tidak punya data kata kunci atau credits. Film yang tidak punya pasangan akan bernilai kosong, lalu diubah menjadi daftar kosong di tahap berikutnya.

In [ ]:
df_gabungan = movies_clean.merge(
    keywords_clean[["id", "keywords"]],
    on="id",
    how="left",
)
df_gabungan = df_gabungan.merge(
    credits_clean[["id", "cast", "crew"]],
    on="id",
    how="left",
)

print("Jumlah baris hasil merge :", len(df_gabungan))
print("Jumlah ID unik           :", df_gabungan["id"].nunique())
print("\nNilai kosong setelah merge:")
print(df_gabungan[["keywords", "cast", "crew"]].isna().sum())


## 11. Mengekstrak genre, kata kunci, dan sutradara

Kolom `genres`, `keywords`, dan `crew` di file asli masih berupa teks seperti daftar JSON, misalnya `[{'id': 18, 'name': 'Drama'}, ...]`. Tahap ini mengambil namanya saja, sehingga diperoleh:

- `genres_list`: daftar nama genre
- `keywords_list`: daftar kata kunci
- `director`: nama kru dengan job Director. Nama sutradara yang berulang pada film yang sama dibuang, dengan tetap mempertahankan urutan kemunculannya.

Teks yang kosong atau tidak bisa dibaca diubah menjadi daftar kosong agar tidak error. Saat disimpan ke CSV, daftar ini ditulis sebagai teks yang dipisah tanda `|`.


In [ ]:
def parse_obj_list(value):
    if pd.isna(value):
        return []
    try:
        hasil = ast.literal_eval(value)
        return hasil if isinstance(hasil, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []


def extract_genres(value):
    data = parse_obj_list(value)
    return [
        item["name"]
        for item in data
        if isinstance(item, dict) and "name" in item
    ]


def extract_keywords(value):
    data = parse_obj_list(value)
    return [
        item["name"]
        for item in data
        if isinstance(item, dict) and "name" in item
    ]


def extract_director(value):
    data = parse_obj_list(value)
    nama = [
        item["name"]
        for item in data
        if isinstance(item, dict)
        and item.get("job") == "Director"
        and item.get("name")
    ]
    # Membuang nama sutradara yang berulang, dengan tetap mempertahankan urutan
    return list(dict.fromkeys(nama))


df_gabungan["genres_list"] = df_gabungan["genres"].apply(extract_genres)
df_gabungan["keywords_list"] = df_gabungan["keywords"].apply(extract_keywords)
df_gabungan["director"] = df_gabungan["crew"].apply(extract_director)

display(
    df_gabungan[["title", "genres_list", "keywords_list", "director"]].head(10)
)


## 12. Eksplorasi data: genre

Sebelum data disimpan, isi data dilihat polanya terlebih dahulu, dimulai dari genre. Satu film bisa punya lebih dari satu genre, jadi total kemunculan genre lebih banyak dari jumlah film. Sel berikut menampilkan sepuluh genre yang paling sering muncul.


In [ ]:
genre_counter = Counter(
    genre for genres in df_gabungan["genres_list"] for genre in genres
)
df_genre = (
    pd.DataFrame(genre_counter.items(), columns=["Genre", "Jumlah Film"])
    .sort_values(by="Jumlah Film", ascending=False)
)

print("10 Genre Film Terbanyak:")
display(df_genre.head(10))

plt.figure(figsize=(10, 6))
plt.barh(
    df_genre.head(10)["Genre"][::-1],
    df_genre.head(10)["Jumlah Film"][::-1],
)
plt.title("10 Genre Film Terbanyak")
plt.xlabel("Jumlah Film")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()


## 13. Eksplorasi data: tahun rilis

Tanggal rilis diubah ke format tanggal, lalu diambil tahunnya. Tanggal yang tidak valid membuat tahun rilis kosong, tetapi filmnya tidak dihapus. Grafik pada sel berikut menunjukkan jumlah film per tahun.


In [ ]:
df_gabungan["release_date"] = pd.to_datetime(
    df_gabungan["release_date"],
    errors="coerce",
)
df_gabungan["release_year"] = df_gabungan["release_date"].dt.year

df_tahun = df_gabungan["release_year"].value_counts().sort_index()

print("10 Tahun dengan Jumlah Film Terbanyak:")
display(
    df_tahun.sort_values(ascending=False)
    .head(10)
    .rename_axis("Tahun")
    .reset_index(name="Jumlah Film")
)

plt.figure(figsize=(12, 5))
plt.plot(df_tahun.index, df_tahun.values)
plt.title("Distribusi Film Berdasarkan Tahun Rilis")
plt.xlabel("Tahun Rilis")
plt.ylabel("Jumlah Film")
plt.grid(True)
plt.tight_layout()
plt.show()

print(
    "Jumlah tanggal rilis kosong/tidak valid:",
    int(df_gabungan["release_year"].isna().sum()),
)


## 14. Eksplorasi data: rating dan jumlah vote

Tahap ini melihat dua kolom angka:

- `vote_average`: rating rata-rata TMDB, skala 0 sampai 10
- `vote_count`: jumlah orang yang memberi rating

Kedua kolom ini membantu memahami kualitas data, walaupun model content-based utamanya memakai teks, yaitu sinopsis, genre, kata kunci, dan sutradara.

Hal yang perlu diperhatikan:

- Banyak film hanya punya sedikit vote, jadi rata-rata `vote_count` jauh lebih besar dari nilai tengahnya.
- Rating 0 biasanya berarti film belum punya vote, bukan berarti filmnya buruk. Film dengan 0 vote berjumlah 2.896 dan ratingnya 0, sehingga akan terlihat sebagai "0,0" di antarmuka. Tim perlu memutuskan apakah nilai tersebut ditampilkan sebagai "belum ada rating". Dataset tidak diubah untuk kasus ini.
- Rating 10 biasanya berasal dari sangat sedikit vote, jadi belum tentu film terbaik.
- Tiga film tanpa judul yang nanti dihapus tidak punya rating, jadi angka di grafik ini sama dengan dataset final.


In [ ]:
rating = df_gabungan["vote_average"]
jumlah_vote = df_gabungan["vote_count"]

print("===== STATISTIK DESKRIPTIF vote_average =====")
print(rating.describe())

print("\n===== STATISTIK DESKRIPTIF vote_count =====")
print(jumlah_vote.describe())

n_nol_vote = int((jumlah_vote == 0).sum())
n_rating_nol = int((rating == 0).sum())
n_keduanya = int(((jumlah_vote == 0) & (rating == 0)).sum())
n_rating_nol_ada_vote = int(((rating == 0) & (jumlah_vote > 0)).sum())

print(f"\nFilm dengan 0 vote             : {n_nol_vote}")
print(f"Film dengan rating 0           : {n_rating_nol}")
print(f"  di antaranya tanpa vote      : {n_keduanya}")
print(f"  tetapi punya vote            : {n_rating_nol_ada_vote}")
print("Korelasi Pearson rating vs jumlah vote:", rating.corr(jumlah_vote))

# Histogram rating
plt.figure(figsize=(10, 5))
plt.hist(rating.dropna(), bins=20, range=(0, 10), edgecolor="black")
plt.title("Distribusi Rating Film (vote_average)")
plt.xlabel("Rating (0–10)")
plt.ylabel("Jumlah Film")
plt.tight_layout()
plt.show()

# Histogram jumlah vote: skala biasa dan log(1 + vote_count)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(jumlah_vote.dropna(), bins=40, edgecolor="black")
axes[0].set_title("Distribusi Jumlah Vote")
axes[0].set_xlabel("vote_count")
axes[0].set_ylabel("Jumlah Film")

axes[1].hist(np.log1p(jumlah_vote.dropna()), bins=30, edgecolor="black")
axes[1].set_title("Distribusi log(1 + vote_count)")
axes[1].set_xlabel("log(1 + vote_count)")
axes[1].set_ylabel("Jumlah Film")
plt.tight_layout()
plt.show()

# Hubungan rating dan jumlah vote
plt.figure(figsize=(10, 6))
plt.scatter(jumlah_vote, rating, alpha=0.15, s=10)
plt.title("Rating vs Jumlah Vote")
plt.xlabel("vote_count")
plt.ylabel("vote_average")
plt.tight_layout()
plt.show()


## 15. Pemeriksaan setelah data dirapikan

Sebelum tahap akhir, hasil dirapikan diperiksa ulang:

- apakah masih ada ID duplikat
- jumlah nilai kosong pada kolom penting
- jumlah film tanpa genre, kata kunci, atau sutradara (film ini tetap disimpan)

Untuk `overview`, pemeriksaan dengan `.isna()` saja tidak cukup, karena NaN sudah diisi teks kosong sehingga hasilnya nol. Pemeriksaan yang benar memakai fungsi `jumlah_teks_kosong` dan hasilnya harus 959.


In [ ]:
print("Ukuran dataset :", df_gabungan.shape)
print("ID duplikat    :", int(df_gabungan["id"].duplicated().sum()))

kolom_penting = [
    "id",
    "title",
    "overview",
    "genres_list",
    "keywords_list",
    "director",
    "release_year",
]

print("\nNilai kosong (NaN) pada kolom penting:")
print(df_gabungan[kolom_penting].isna().sum())
print(
    "Overview kosong (NaN/teks kosong/hanya spasi):",
    jumlah_teks_kosong(df_gabungan["overview"]),
)

print("\nFilm tanpa genre    :", int((df_gabungan["genres_list"].apply(len) == 0).sum()))
print("Film tanpa keyword  :", int((df_gabungan["keywords_list"].apply(len) == 0).sum()))
print("Film tanpa director :", int((df_gabungan["director"].apply(len) == 0).sum()))


## 16. Menghapus film tanpa judul

Ada film dengan ID valid, tetapi kolom `title`-nya kosong. Judul dibutuhkan untuk menampilkan hasil rekomendasi, jadi baris ini dihapus. Film-film ini punya sinopsis, sehingga jumlah overview kosong tetap 959.


In [ ]:
film_tanpa_judul = df_gabungan[df_gabungan["title"].isna()]
print("Jumlah film tanpa judul:", len(film_tanpa_judul))
display(
    film_tanpa_judul[
        ["id", "title", "release_date", "overview", "genres_list"]
    ]
)

df_gabungan = df_gabungan.dropna(subset=["title"]).copy()
df_gabungan = df_gabungan.reset_index(drop=True)

print("\nJumlah baris setelah pembersihan judul:", len(df_gabungan))
print("Judul kosong :", int(df_gabungan["title"].isna().sum()))
print("ID duplikat  :", int(df_gabungan["id"].duplicated().sum()))
print(
    "Overview kosong setelah judul dihapus:",
    jumlah_teks_kosong(df_gabungan["overview"]),
)
print(
    "Tahun rilis kosong:",
    int(df_gabungan["release_year"].isna().sum()),
)


## 17. Pemeriksaan kolom adult, status, dan film tanpa vote

Kolom `adult` dan `status` tidak dipakai sebagai penyaring, sehingga film dewasa atau film yang belum rilis tetap bisa masuk ke dataset akhir. Sel berikut menampilkan jumlahnya. Tim perlu memutuskan apakah data tersebut perlu disaring.

Catatan untuk tampilan antarmuka:

- Film dengan 0 vote memiliki rating 0. Nilai ini akan terlihat sebagai "0,0" di antarmuka.
- Tim perlu memutuskan apakah nilai tersebut ditampilkan sebagai "belum ada rating".
- Dataset tidak diubah untuk kasus ini.


In [ ]:
print("===== SEBARAN NILAI adult =====")
print(df_gabungan["adult"].value_counts(dropna=False))

print("\n===== SEBARAN NILAI status =====")
print(df_gabungan["status"].value_counts(dropna=False))

n_dewasa = int(
    df_gabungan["adult"].astype(str).str.strip().str.lower().eq("true").sum()
)
n_bukan_rilis = int((df_gabungan["status"] != "Released").sum())
n_status_kosong = int(df_gabungan["status"].isna().sum())
n_nol_vote = int((df_gabungan["vote_count"] == 0).sum())

print(f"\nFilm dewasa (adult=True)           : {n_dewasa}")
print(f"Film dengan status bukan Released  : {n_bukan_rilis}")
print(f"  di antaranya status kosong       : {n_status_kosong}")
print(f"Film dengan 0 vote                 : {n_nol_vote}")


## 18. Menyimpan dataset bersih

Kolom yang disimpan hanya yang sudah disepakati tim, dengan urutan:
`id`, `title`, `overview`, `genres_list`, `keywords_list`, `director`, `release_year`, `vote_average`, `vote_count`.

Daftar genre dan kata kunci diubah menjadi teks yang dipisah tanda `|` agar mudah dibaca di CSV dan bisa diimpor ke MySQL.

Sebelum disimpan, `release_year` dan `vote_count` diubah menjadi bilangan bulat yang boleh kosong, agar sesuai skema MySQL (`SMALLINT` dan `INT`). Spasi pada sinopsis juga dibersihkan, lalu sinopsis kosong disimpan sebagai nilai kosong supaya hitungan overview kosong tetap 959 saat CSV dibaca ulang.

File `movies_clean_final.csv` dibagikan ke tim lewat Google Drive, tidak diunggah ke GitHub. Jika nama atau isi kolom akan diubah, kabari Orang 2 dan Orang 3 terlebih dahulu.


In [ ]:
KOLOM_FINAL = [
    "id",
    "title",
    "overview",
    "genres_list",
    "keywords_list",
    "director",
    "release_year",
    "vote_average",
    "vote_count",
]

df_final = df_gabungan[KOLOM_FINAL].copy()

for kolom in ["genres_list", "keywords_list", "director"]:
    df_final[kolom] = df_final[kolom].apply(
        lambda x: "|".join(x) if isinstance(x, list) else ""
    )

# Mengubah tahun rilis dan jumlah vote menjadi bilangan bulat (boleh kosong) agar sesuai skema MySQL
df_final["release_year"] = df_final["release_year"].astype("Int64")
df_final["vote_count"] = df_final["vote_count"].astype("Int64")

# Membersihkan spasi di awal/akhir sinopsis, lalu mengubah sinopsis kosong menjadi NaN
df_final["overview"] = df_final["overview"].str.strip().replace("", pd.NA)

df_final.to_csv(CSV_OUTPUT, index=False)

print("Dataset berhasil disimpan ke:", CSV_OUTPUT.resolve())
print("Ukuran dataset:", df_final.shape)
display(df_final.head())

# ----- Validasi akhir -----
print("\n===== VALIDASI AKHIR =====")
print("Kolom sesuai kontrak      :", list(df_final.columns) == KOLOM_FINAL)
print("Kolom yang tersimpan      :", list(df_final.columns))
print("Jumlah baris              :", len(df_final))
print("ID unik                   :", df_final["id"].nunique())
print("ID duplikat               :", int(df_final["id"].duplicated().sum()))
print("Judul kosong              :", int(df_final["title"].isna().sum()))
print(
    "Overview kosong (NaN/teks kosong/hanya spasi):",
    jumlah_teks_kosong(df_final["overview"]),
)
print("Tahun rilis kosong        :", int(df_final["release_year"].isna().sum()))
print("Rating kosong             :", int(df_final["vote_average"].isna().sum()))
print("Jumlah vote kosong        :", int(df_final["vote_count"].isna().sum()))

assert list(df_final.columns) == KOLOM_FINAL, "Urutan/nama kolom tidak sesuai kontrak."
assert df_final["id"].duplicated().sum() == 0, "Masih ada ID duplikat."
assert df_final["title"].isna().sum() == 0, "Masih ada judul kosong."
assert jumlah_teks_kosong(df_final["overview"]) == 959, (
    "Hitungan overview kosong seharusnya 959."
)
print("\nValidasi akhir berhasil.")

# Memeriksa ulang file CSV: tahun dan jumlah vote tanpa desimal, overview kosong 959
cek = pd.read_csv(CSV_OUTPUT, dtype={"release_year": "Int64", "vote_count": "Int64"})
print("\n===== PEMERIKSAAN ULANG CSV =====")
print(cek[["release_year", "vote_count", "overview"]].head())
print(
    "Overview kosong (NaN) saat CSV dibaca ulang:",
    int(cek["overview"].isna().sum()),
)

# Cuplikan nilai mentah di file CSV (tanpa inferensi tipe pandas)
import csv as csv_modul

with Path(CSV_OUTPUT).open(encoding="utf-8", newline="") as f_csv:
    pembaca = csv_modul.DictReader(f_csv)
    print("\nLima baris mentah release_year dan vote_count pada CSV:")
    for i, baris in enumerate(pembaca):
        if i >= 5:
            break
        print(f"  release_year={baris['release_year']!r}  vote_count={baris['vote_count']!r}")

assert int(cek["overview"].isna().sum()) == 959, (
    "Hitungan overview kosong pada CSV seharusnya 959."
)
print("Pemeriksaan ulang CSV berhasil.")


## 19. Selesai

Notebook ini menghasilkan `movies_clean_final.csv` berisi 45.430 film yang siap dipakai untuk model dan backend. File ini dibagikan ke tim lewat Google Drive, tidak lewat GitHub.

Langkah Data selanjutnya dikerjakan terpisah:

- membuat `scripts/import_csv.py` bersama Orang 3 sebagai pembuat tabel MySQL
- membantu menyiapkan data uji untuk evaluasi

Di Google Colab, sel berikut akan mengunduh file hasilnya. Di komputer sendiri, file sudah tersimpan di folder kerja.


In [ ]:
if Path(CSV_OUTPUT).exists():
    print("File hasil:", Path(CSV_OUTPUT).resolve())
    try:
        from google.colab import files

        files.download(str(CSV_OUTPUT))
    except ImportError:
        print("Bukan lingkungan Colab; file tidak diunduh otomatis.")
else:
    raise FileNotFoundError(f"{CSV_OUTPUT} belum dibuat. Jalankan sel penyimpanan lebih dulu.")
